# Step 4 — Reconstruction Models

This notebook documents the two reconstruction methods:

1. **Kalman Filter** — physics-based constant-velocity smoother
2. **BiGRU** — learned sequence model trained in `noel_part/model_training.ipynb`

Both are implemented in `noel_part/reconstruction.py` and used directly from there.

## Cell 1 — Setup

In [1]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ── Point to noel_part ────────────────────────────────────────────────────────
NOEL_DIR = Path(r"C:\Users\marko\Desktop\AeroML3\noel_part")
# If path above is wrong, find it automatically:
if not NOEL_DIR.exists():
    for _candidate in [Path("../noel_part"), Path("../../noel_part")]:
        if _candidate.resolve().exists():
            NOEL_DIR = _candidate.resolve()
            break

os.chdir(NOEL_DIR)
sys.path.insert(0, str(NOEL_DIR))

BASE_DIR  = Path(".")
CLEAN_DIR = BASE_DIR / "cleaned_data_final"
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

print(f"Working dir : {os.getcwd()}")
print(f"Flights available: {len([f for s in sorted(CLEAN_DIR.iterdir()) if s.is_dir() for f in s.iterdir() if f.is_dir()])}")

c:\Users\marko\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Working dir : C:\Users\marko\Desktop\AeroML3\noel_part
Flights available: 2149


## Cell 2 — Kalman Filter

The Kalman filter is a physics-based approach. It does not require training data.

### How it works

1. **Warmup** — runs through the last 15 ADS-B points before the gap, estimating position and velocity via predict + update steps
2. **Override velocity** — sets the velocity to point directly from the last known position to the first known position after the gap, ensuring a straight path
3. **Forward predict** — propagates the state forward across the gap using a constant-velocity motion model
4. **Retime** — rescales timestamps so the aircraft moves at constant speed, fixing drift

### State vector
```
x = [latitude, longitude, altitude, dlat/s, dlon/s, dalt/s]
```

### Key matrices
- **F** — state transition (position += velocity × dt)
- **Q** — process noise (uncertainty in acceleration)
- **R** — measurement noise (ADS-B accuracy ~10m lat/lon, ~50m alt)

In [2]:
from reconstruction import reconstruct_gap_kalman, compute_features

# ── Load a sample flight ──────────────────────────────────────────────────────
flights = [f for s in sorted(CLEAN_DIR.iterdir()) if s.is_dir()
           for f in sorted(s.iterdir()) if f.is_dir()]

# Use the Ireland→New York flight
FLIGHT_NAME = "20230709_a67811_122827_132249"
FLIGHT_DIR  = next((s/FLIGHT_NAME for s in CLEAN_DIR.iterdir()
                    if s.is_dir() and (s/FLIGHT_NAME).exists()), None)

if FLIGHT_DIR is None:
    FLIGHT_DIR = flights[0]
    print(f"Using fallback flight: {FLIGHT_DIR.name}")
else:
    print(f"Using flight: {FLIGHT_NAME}")

bef  = pd.read_parquet(FLIGHT_DIR / "adsb_before.parquet")
aft  = pd.read_parquet(FLIGHT_DIR / "adsb_after.parquet")
adsc = pd.read_parquet(FLIGHT_DIR / "adsc.parquet")

for df in [bef, aft, adsc]:
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_localize(None)
    df.sort_values("timestamp", inplace=True)
    df.reset_index(drop=True, inplace=True)

# Trim to gap boundary
t_gap_start = bef["timestamp"].iloc[-1]
t_gap_end   = aft["timestamp"].iloc[0]
bef_trim = bef[bef["timestamp"] >= t_gap_start - pd.Timedelta(minutes=30)].reset_index(drop=True)
aft_trim = aft[aft["timestamp"] <= t_gap_end   + pd.Timedelta(minutes=30)].reset_index(drop=True)

gap_min = (t_gap_end - t_gap_start).total_seconds() / 60
print(f"Gap duration : {gap_min:.1f} min")
print(f"Before points: {len(bef_trim)}")
print(f"After points : {len(aft_trim)}")

# Run Kalman
recon_kalman = compute_features(reconstruct_gap_kalman(bef_trim, aft_trim))
print(f"\nKalman reconstruction: {len(recon_kalman)} steps")
print(f"Start: lat={recon_kalman['latitude'].iloc[0]:.3f}  lon={recon_kalman['longitude'].iloc[0]:.3f}")
print(f"End  : lat={recon_kalman['latitude'].iloc[-1]:.3f}  lon={recon_kalman['longitude'].iloc[-1]:.3f}")
display(recon_kalman[["timestamp","latitude","longitude","altitude"]].head(5))

Using flight: 20230709_a67811_122827_132249
Gap duration : 212.0 min
Before points: 121
After points : 121

Kalman reconstruction: 848 steps
Start: lat=54.982  lon=-14.279
End  : lat=48.167  lon=-61.677


,timestamp,latitude,longitude,altitude
0,2023-07-09 12:02:45,54.982206,-14.279015,10974.292584
1,2023-07-09 12:03:00,54.973537,-14.339309,10975.066143
2,2023-07-09 12:03:15,54.964869,-14.399591,10975.839547
3,2023-07-09 12:03:30,54.956203,-14.459861,10976.612795
4,2023-07-09 12:03:45,54.947540,-14.520119,10977.385887


## Cell 3 — BiGRU Model

The BiGRU is a learned sequence model trained on complete ADS-B trajectory segments.

### Architecture
```
Input : (batch=1, sequence=10, features=7)
         features = [latitude, longitude, altitude, vx, vy, acceleration, turn_rate]
  ↓
Bidirectional GRU (hidden=128, layers=2)
  ↓
Attention pooling over sequence
  ↓
FC: 256 → 128 → ReLU → Dropout → 3
Output: (lat_residual, lon_residual, alt_residual)
```

### Key design choices
- **Residual prediction** — model predicts a correction on top of great-circle extrapolation, not absolute position
- **Relative coordinates** — lat/lon/alt made relative to last known point before prediction
- **Bidirectional** — uses both forward and backward context from the ADS-B segments
- **Attention** — weights each timestep's contribution to the final prediction

### Training
The model was trained in `noel_part/model_training.ipynb` on complete ADS-B segments.
Training data: sliding windows of 10 ADS-B points → predict next point.
Loss: MSE on residual (predicted - great-circle extrapolation).

In [ ]:
from reconstruction import TrajectoryBiGRU, FEATURE_COLS, TARGET_COLS, SEQUENCE_LEN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TrajectoryBiGRU(
    input_size  = len(FEATURE_COLS),
    hidden_size = 128,
    num_layers  = 2,
    output_size = len(TARGET_COLS),
    dropout     = 0.3
).to(device)
model.load_state_dict(torch.load("models/BiGRU.pth", map_location=device))
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Device         : {device}")
print(f"Parameters     : {n_params:,}")
print(f"Input features : {FEATURE_COLS}")
print(f"Output targets : {TARGET_COLS}")
print(f"Sequence length: {SEQUENCE_LEN}")
print()
print("Model architecture:")
print(model)

## Cell 4 — Run BiGRU on sample flight

In [ ]:
from reconstruction import reconstruct_gap, compute_features

recon_bigru = compute_features(
    reconstruct_gap(model, bef_trim, aft_trim, FEATURE_COLS, TARGET_COLS, SEQUENCE_LEN, device)
)

print(f"BiGRU reconstruction: {len(recon_bigru)} steps")
print(f"Start: lat={recon_bigru['latitude'].iloc[0]:.3f}  lon={recon_bigru['longitude'].iloc[0]:.3f}")
print(f"End  : lat={recon_bigru['latitude'].iloc[-1]:.3f}  lon={recon_bigru['longitude'].iloc[-1]:.3f}")
display(recon_bigru[["timestamp","latitude","longitude","altitude"]].head(5))

## Cell 5 — Compare Kalman vs BiGRU on this flight

In [ ]:
from reconstruction import reconstruct_gap_baseline
from pyproj import Geod
from scipy.interpolate import interp1d

geod = Geod(ellps="WGS84")

def reconstruct_forward_only(before_df, after_df, dt=15.0):
    last_time  = before_df["timestamp"].iloc[-1]
    first_time = after_df["timestamp"].iloc[0]
    n_steps = max(1, int(round((first_time-last_time).total_seconds()/dt)))
    lat0=float(before_df["latitude"].iloc[-1]); lon0=float(before_df["longitude"].iloc[-1])
    alt0=float(before_df["altitude"].iloc[-1])
    lat1=float(after_df["latitude"].iloc[0]);   lon1=float(after_df["longitude"].iloc[0])
    alt1=float(after_df["altitude"].iloc[0])
    pts=geod.npts(lon0,lat0,lon1,lat1,n_steps)
    lats=np.array([p[1] for p in pts]); lons=np.array([p[0] for p in pts])
    alts=np.linspace(alt0,alt1,n_steps)
    timestamps=[last_time+pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    result=pd.DataFrame({"latitude":lats,"longitude":lons,"altitude":alts})
    result["timestamp"]=timestamps; result["interpolated"]=True
    return result

def retime_to_constant_speed(recon_df, before_df, after_df, dt=15.0):
    lats=recon_df["latitude"].values; lons=recon_df["longitude"].values
    alts=recon_df["altitude"].values if "altitude" in recon_df.columns else np.zeros(len(recon_df))
    cum_dist=np.zeros(len(lats))
    for i in range(1,len(lats)):
        _,_,d=geod.inv(float(lons[i-1]),float(lats[i-1]),float(lons[i]),float(lats[i]))
        cum_dist[i]=cum_dist[i-1]+abs(d)
    total_dist=cum_dist[-1]
    if total_dist==0: return recon_df
    last_time=before_df["timestamp"].iloc[-1]; first_time=after_df["timestamp"].iloc[0]
    total_sec=(first_time-last_time).total_seconds()
    n_steps=max(1,int(round(total_sec/dt)))
    time_fracs=np.clip([dt*(i+1)/total_sec for i in range(n_steps)],0,1)
    old_fracs=cum_dist/total_dist
    f_la=interp1d(old_fracs,lats,bounds_error=False,fill_value=(lats[0],lats[-1]))
    f_lo=interp1d(old_fracs,lons,bounds_error=False,fill_value=(lons[0],lons[-1]))
    f_alt=interp1d(old_fracs,alts,bounds_error=False,fill_value=(alts[0],alts[-1]))
    new_ts=[last_time+pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    return pd.DataFrame({"latitude":f_la(time_fracs),"longitude":f_lo(time_fracs),
                         "altitude":f_alt(time_fracs),"timestamp":new_ts,"interpolated":True})

def error_km(recon_df, truth_df):
    if len(truth_df)==0 or len(recon_df)==0: return float("nan")
    def hav(la1,lo1,la2,lo2):
        la1,lo1,la2,lo2=map(np.radians,[la1,lo1,la2,lo2])
        dlat,dlon=la2-la1,lo2-lo1
        a=np.sin(dlat/2)**2+np.cos(la1)*np.cos(la2)*np.sin(dlon/2)**2
        return 2*6_371_000*np.arcsin(np.sqrt(a))
    errs=[]
    for _,row in truth_df.iterrows():
        dists=hav(row["latitude"],row["longitude"],recon_df["latitude"].values,recon_df["longitude"].values)
        errs.append(dists.min())
    return np.mean(errs)/1000

adsc_gap = adsc[(adsc["timestamp"]>=t_gap_start)&(adsc["timestamp"]<=t_gap_end)].reset_index(drop=True)

recon_base        = reconstruct_forward_only(bef_trim, aft_trim)
recon_kalman_rt   = retime_to_constant_speed(recon_kalman, bef_trim, aft_trim)
recon_bigru_rt    = retime_to_constant_speed(recon_bigru,  bef_trim, aft_trim)

base_err   = error_km(recon_base,     adsc_gap)
kalman_err = error_km(recon_kalman_rt, adsc_gap)
bigru_err  = error_km(recon_bigru_rt,  adsc_gap)

print(f"ADS-C waypoints used for evaluation: {len(adsc_gap)}")
print()
print(f"{'Method':<25} {'Mean error (km)':>16}")
print("-"*43)
print(f"  {'Baseline (great-circle)':<23} {base_err:>14.1f} km")
print(f"  {'Kalman filter':<23} {kalman_err:>14.1f} km")
print(f"  {'BiGRU model':<23} {bigru_err:>14.1f} km")
print()
best = min([("Baseline",base_err),("Kalman",kalman_err),("BiGRU",bigru_err)], key=lambda x:x[1])
print(f"Best method on this flight: {best[0]} ({best[1]:.1f} km)")

## Cell 6 — Retrain BiGRU (optional)

To retrain the model on new data, open and run all cells in:
```
noel_part/model_training.ipynb
```
The saved model will be written to `noel_part/models/BiGRU.pth`.

Note: The current model was trained on short ADS-B segments. For better oceanic gap reconstruction, it would benefit from retraining specifically on oceanic crossing data.